# Chuyên đề 01 · Thực hành chữa gradient biến mất: chạy lại một mạng sigmoid 7 tầng và chín cách chữa bằng PyTorch

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd01-thuc-hanh-gradient-bien-mat.ipynb)

Notebook đi kèm [Chuyên đề 01 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/thuc-hanh-gradient-bien-mat/) (reel nằm ở trang chuyên đề). Nó dựa trên tài liệu
*"Tutorial: Vanishing Gradient"* của Nguyễn Thọ Anh Khoa, Nguyễn Phúc Thịnh, Dương Đình Thắng và Nguyễn Anh Khôi:
một MLP 7 tầng ẩn dùng sigmoid, huấn luyện trên Fashion-MNIST, không học được gì; rồi chín cách chữa. Ở đây ta chạy
lại **đúng mã PyTorch của tài liệu**, đo những thứ tài liệu chỉ tả bằng lời, và kiểm tra lời giải thích của nó.

**Bạn sẽ làm:**
1. Chạy mạng gốc của tài liệu và đo gradient ở từng tầng: tầng 1 nhận gradient nhỏ hơn tầng ra khoảng hai triệu lần.
2. Kiểm tra lời giải thích "sigmoid bão hòa", và tìm ra thủ phạm thật: trọng số khởi tạo quá nhỏ.
3. Quét độ lệch chuẩn khởi tạo: nhỏ quá thì gradient biến mất, lớn quá thì sigmoid bão hòa.
4. Chạy vài epoch của BatchNorm và của "Shortcut Back" có và không có hook, rồi đọc kết quả 100 epoch × 3 seed của
   cả 15 thí nghiệm.

**Cần biết trước:** [Bài 09 · Gradient biến mất & khởi tạo](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vanishing-gradient/)
(tích của các thừa số $w \cdot f'$), [Bài 07 · Momentum, Adam & AdamW](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/),
[Bài 10 · BatchNorm](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/chuan-hoa/) và
[Bài 13 · Kết nối tắt](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/resnet/).
**Thời gian chạy:** khoảng 1 phút trên CPU, kể cả tải Fashion-MNIST (~30 MB) lần đầu.

Khác với các notebook của từng bài (numpy từ đầu), chuyên đề này dùng **PyTorch**, vì mục đích là chạy lại chính mã
của tài liệu. Các ô huấn luyện ngắn chạy trực tiếp; kết quả 100 epoch × 3 seed (45 lần huấn luyện, khoảng nửa giờ)
được nạp từ đúng tệp mà reel dùng. Notebook kiểm tra rằng các epoch nó tự chạy trùng với epoch đầu của các lần chạy
dài đó.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

PyTorch, numpy và matplotlib (Colab đã cài sẵn). Ta dùng 2 luồng CPU, như máy Colab miễn phí và như các lần chạy
dài: số luồng khác thì phép cộng dấu phẩy động cộng theo thứ tự khác, và các chữ số cuối lệch đi.
Hàm `vn()` in số kiểu Việt Nam (dấu phẩy thập phân); `num()` in số rất nhỏ dạng $3{,}3 \cdot 10^{-9}$.

In [ ]:
import gzip
import hashlib
import json
import math
import os
import random
import urllib.request
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim  # noqa: PLR0402  (the tutorial's import, as printed)
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
device = torch.device("cpu")  # các con số dưới đây là của CPU; GPU cộng theo thứ tự khác

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def num(x, sig=3):
    """sig chữ số có nghĩa: 0,243 · 0,0884 · 55,7 · 6,60·10⁻³ · 3,27·10⁻⁹"""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)))
    if -2 <= e < 4:
        return vn(x, max(0, sig - 1 - e))
    return f"{vn(x / 10**e, sig - 1)}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Nhãn trục kiểu Việt Nam; gọi ngay trước plt.show()."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            step = max(1, math.ceil((k1 - k0) / 8))
            axis.set_major_locator(FixedLocator([10.0**k for k in range(k0, k1 + 1) if k % step == 0]))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))

## 2 · Dữ liệu: Fashion-MNIST

Fashion-MNIST (Xiao, Rasul & Vollgraf, 2017) có cùng khuôn với MNIST: 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra,
mỗi ảnh xám 28 × 28, nhưng là 10 loại quần áo và giày dép thay cho chữ số. Ta tải bốn tệp gốc từ repo của bộ dữ liệu
và kiểm tra mã MD5 (cùng mã mà torchvision dùng).

Tài liệu gốc nạp dữ liệu bằng `torchvision.datasets.FashionMNIST(..., transform=transforms.ToTensor())`. `ToTensor()`
chỉ chia mỗi byte cho 255, nên ta đọc thẳng các byte rồi chia cho 255: cùng những con số đó, nhưng nhanh hơn nhiều,
vì không phải dựng một ảnh PIL cho mỗi mẫu ở mỗi epoch.

In [ ]:
URL = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
FILES = {  # mã MD5 chuẩn của Fashion-MNIST
    "train-images-idx3-ubyte.gz": "8d4fb7e6c68d591d4c3dfef9ec88bf0d",
    "train-labels-idx1-ubyte.gz": "25c81989df183df01b3e8a0aad5dffbe",
    "t10k-images-idx3-ubyte.gz": "bef4ecab320f06d8554ea6380940ec79",
    "t10k-labels-idx1-ubyte.gz": "bb300cfdad3c16e7a12a480ee83cd310",
}
DATA = Path("fashion")
DATA.mkdir(exist_ok=True)
for name, md5 in FILES.items():
    p = DATA / name
    if not p.exists():
        urllib.request.urlretrieve(URL + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


RUNS = fetch_json("vg-tutorial.json", "3d844c521e5ab90fb7a5c39401852ce9")

In [ ]:
CLASSES = ["Áo thun", "Quần", "Áo len", "Váy", "Áo khoác", "Dép", "Áo sơ mi", "Giày thể thao", "Túi", "Bốt"]


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        x = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        y = np.frombuffer(f.read(), np.uint8, offset=8)
    return torch.from_numpy(x.astype(np.float32) / 255.0), torch.from_numpy(y.astype(np.int64))


X_all, y_all = load("train")
X_test_all, y_test_all = load("t10k")
print("huấn luyện:", tuple(X_all.shape), " kiểm tra:", tuple(X_test_all.shape))

fig, axes = plt.subplots(1, 10, figsize=(11, 1.6))
for c, ax in enumerate(axes):
    i = int((y_test_all == c).nonzero()[0, 0])
    ax.imshow(X_test_all[i, 0], cmap="gray_r")
    ax.set_title(CLASSES[c], fontsize=8)
    ax.axis("off")
plt.show()

Mỗi ô là ảnh đầu tiên của một lớp trong tập kiểm tra, vẽ nét tối trên nền sáng; tên lớp ở phía trên.

## 3 · Mạng gốc của tài liệu

Đây là mã của tài liệu, chép nguyên văn: hàm `set_seed`, cách chia 90/10 bằng `random_split`, các `DataLoader`
với `batch_size = 512` (tài liệu dùng biến này mà không gán; 512 là giá trị trong notebook lời giải của nó), và mạng
784 → 7 × 128 → 10, mỗi tầng ẩn một sigmoid, trọng số khởi tạo $\mathcal{N}(0;\ 0{,}05^2)$, bias bằng 0.

Có một chỗ đổi: `Dataset` là các tensor đã chia cho 255 ở trên (`TensorDataset`) thay cho `FashionMNIST` + `ToTensor()`.
Cùng ảnh, cùng thứ tự xáo trộn (vẫn là `DataLoader`), cùng phép tính; chỉ bỏ bước dựng ảnh PIL.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


SEED = 42
batch_size = 512
train_dataset = TensorDataset(X_all, y_all)
test_dataset = TensorDataset(X_test_all, y_test_all)


def make_loaders(seed=SEED):
    """Tài liệu: set_seed(SEED), rồi random_split 90/10 và ba DataLoader."""
    set_seed(seed)
    train_size = int(len(train_dataset) * 0.9)
    train_subset, val_subset = random_split(train_dataset, [train_size, len(train_dataset) - train_size])
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


class MLP(nn.Module):
    def __init__(self, input_dims, hidden_dims, output_dims, std=0.05):
        super(MLP, self).__init__()  # noqa: UP008  (the tutorial's code, as printed)
        self.layer1 = nn.Linear(input_dims, hidden_dims)
        self.layer2 = nn.Linear(hidden_dims, hidden_dims)
        self.layer3 = nn.Linear(hidden_dims, hidden_dims)
        self.layer4 = nn.Linear(hidden_dims, hidden_dims)
        self.layer5 = nn.Linear(hidden_dims, hidden_dims)
        self.layer6 = nn.Linear(hidden_dims, hidden_dims)
        self.layer7 = nn.Linear(hidden_dims, hidden_dims)
        self.output = nn.Linear(hidden_dims, output_dims)

        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=std)  # tài liệu: std=0.05 (mục 5 thử 1.0 và 10.0)
                nn.init.constant_(module.bias, 0.0)

    def forward(self, x):
        x = nn.Flatten()(x)
        x = self.layer1(x)
        x = nn.Sigmoid()(x)
        x = self.layer2(x)
        x = nn.Sigmoid()(x)
        x = self.layer3(x)
        x = nn.Sigmoid()(x)
        x = self.layer4(x)
        x = nn.Sigmoid()(x)
        x = self.layer5(x)
        x = nn.Sigmoid()(x)
        x = self.layer6(x)
        x = nn.Sigmoid()(x)
        x = self.layer7(x)
        x = nn.Sigmoid()(x)
        out = self.output(x)
        return out

Vòng huấn luyện là của tài liệu (SGD, `lr = 1e-2`, `CrossEntropyLoss`), thêm một việc: sau mỗi `loss.backward()`,
ghi lại trung bình $|\partial L / \partial W|$ của từng tầng `Linear`, rồi lấy trung bình qua các batch của epoch.
Đó chính là số liệu mà tài liệu vẽ thành biểu đồ "Gradient Flow" mà không in ra.

In [ ]:
def linears(model):
    return [m for m in model.modules() if isinstance(m, nn.Linear)]


def evaluate(model, loader, criterion):
    model.eval()
    loss = correct = count = 0
    with torch.no_grad():
        for X, y in loader:
            out = model(X)
            loss += criterion(out, y).item()
            correct += (torch.argmax(out, 1) == y).sum().item()
            count += len(y)
    return loss / len(loader), correct / count


def train(model, optimizer, loaders, epochs, before_epoch=None):
    train_loader, val_loader = loaders
    criterion = nn.CrossEntropyLoss()
    hist = []
    for epoch in range(epochs):
        if before_epoch:
            before_epoch(epoch)
        model.train()
        layers = linears(model)
        g = [0.0] * len(layers)
        train_loss = 0.0
        for X_train, y_train in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_train), y_train)
            loss.backward()
            for k, m in enumerate(layers):
                g[k] += m.weight.grad.abs().mean().item()
            optimizer.step()
            train_loss += loss.item()
        _, val_acc = evaluate(model, val_loader, criterion)
        hist.append(
            {
                "train_loss": train_loss / len(train_loader),
                "val_acc": val_acc,
                "grad": [v / len(train_loader) for v in g],
            }
        )
        print(f"epoch {epoch + 1}: train loss {vn(hist[-1]['train_loss'], 4)} · val {pct(val_acc)}")
    return hist


loaders = make_loaders()
model = MLP(784, 128, 10)
base = train(model, optim.SGD(model.parameters(), lr=1e-2), loaders, epochs=2)

Hai epoch này phải trùng với hai epoch đầu của lần chạy dài (100 epoch, cùng mã, cùng seed 42), mà ta nạp từ tệp
kết quả. Trên cùng loại CPU chúng trùng đến từng chữ số; trên GPU hay CPU khác, các chữ số cuối có thể lệch một chút.

In [ ]:
stored = RUNS["runs"]["baseline"]
for e in range(2):
    print(f"epoch {e + 1}: vừa chạy {vn(base[e]['train_loss'], 6)} · lần chạy dài {vn(stored['trainLoss'][e], 6)}")
assert abs(base[0]["train_loss"] - 2.3083373443) < 1e-3, "khác với reel: hãy kiểm tra lại dữ liệu và số luồng"
assert abs(base[0]["val_acc"] - 0.0936667) < 0.005
print(
    f"\nTài liệu in sau 100 epoch: val {pct(RUNS['paper']['baseline']['val'])} · test {pct(RUNS['paper']['baseline']['test'])}"
)
print("Lần chạy dài, 3 seed, test:", ", ".join(pct(v) for v in stored["test"]))

Sau 100 epoch, mạng đoán đúng 10% ảnh kiểm tra, đúng như tài liệu in: bằng đoán mò giữa 10 lớp. Cả ba seed đều thế.

### 🤔 Dự đoán trước
Gradient ở tầng 1 nhỏ hơn gradient ở tầng ra bao nhiêu lần: 10 lần, 1.000 lần, hay hơn một triệu lần?

In [ ]:
g = base[0]["grad"]
names = [f"tầng {k + 1}" for k in range(7)] + ["tầng ra"]
for n, v in zip(names, g):
    print(f"{n:8} {num(v)}")
ratio = g[0] / g[-1]
print(f"\ntầng 1 / tầng ra = {num(ratio)}: nhỏ hơn {vn(1 / ratio / 1e6, 1)} triệu lần")
assert abs(g[0] / 3.27468624e-09 - 1) < 0.01 and abs(g[-1] / 6.60295583e-03 - 1) < 0.01

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(range(7), g[:7], color=BLUE)
ax.bar([7], g[7:], color=ORANGE, hatch="//")
ax.set_yscale("log")
ax.set_xticks(range(8), [str(k + 1) for k in range(7)] + ["ra"])
ax.set(xlabel="tầng (từ đầu vào đến đầu ra)", ylabel="trung bình |∂L/∂W|", title="Gradient từng tầng, epoch 1")
vn_axes(fig)
plt.show()

Hơn hai triệu lần. Trục dọc là thang log: mỗi vạch là một bậc mười. Cột cam gạch chéo là tầng ra; bảy cột xanh là
các tầng ẩn, mỗi cột thấp hơn cột bên phải nó gần một bậc. Tài liệu gọi đúng tên hiện tượng: gradient biến mất. Các
tầng đầu nhận gần như không có tín hiệu sửa lỗi, nên đứng yên.

## 4 · Vì sao? Kiểm tra lời giải thích "sigmoid bão hòa"

Tài liệu giải thích: đạo hàm sigmoid luôn nhỏ hơn 0,25, chồng qua 7 tầng thì tích về 0 — "hiện tượng bão hòa đạo
hàm". Ta đo thẳng: cho một lô ảnh kiểm định đi qua mạng (chưa huấn luyện) và ghi lại, ở mỗi tầng,
- $\sigma'(z)$ trung bình, và tỉ lệ nơ-ron *bão hòa* ($\sigma'(z) < 0{,}01$, tức $|z| > 4{,}6$);
- **độ khác nhau giữa các ảnh**: độ lệch chuẩn của đầu ra theo các ảnh trong lô, trung bình qua các nơ-ron. Nếu nó về
  0, mọi ảnh đến tầng sau như cùng một vectơ.

In [ ]:
def probe(model, X):
    """Ghi lại σ′, tỉ lệ bão hòa và độ khác nhau giữa các ảnh ở đầu ra mỗi sigmoid."""
    rows = []
    h = nn.Flatten()(X)
    with torch.no_grad():
        for m in linears(model)[:-1]:
            h = torch.sigmoid(m(h))
            d = h * (1 - h)
            rows.append((d.mean().item(), (d < 0.01).float().mean().item(), h.std(0).mean().item()))
    return rows


X_probe = loaders[1].dataset[:512][0]  # 512 ảnh kiểm định đầu tiên, như lần chạy dài
set_seed(SEED)
fresh = MLP(784, 128, 10)  # một mạng mới, khởi tạo như mạng gốc (std 0,05), chưa huấn luyện
rows = probe(fresh, X_probe)
print("tầng   σ′ trung bình   bão hòa   khác nhau giữa các ảnh")
for k, (s, sat, sp) in enumerate(rows):
    print(f"{k + 1:4}   {vn(s, 4):>13}   {pct(sat, 0):>7}   {num(sp):>10}")

**Không có nơ-ron nào bão hòa.** $\sigma'$ ở cả 7 tầng vẫn quanh 0,24, gần mức tối đa 0,25: sigmoid đang ở đúng phần
dốc nhất của nó. Thứ sụp đổ là cột cuối: độ khác nhau giữa các ảnh giảm khoảng 7 lần qua mỗi tầng, và đến tầng 7 thì
mọi ảnh gần như cho cùng một vectơ. Lý do: trọng số quá nhỏ. Với $z = xW$ và 128 đầu vào, mỗi tầng nhân độ lệch của
tín hiệu với khoảng $\sigma' \cdot \text{std} \cdot \sqrt{128}$; sigmoid quanh 0 gần như tuyến tính, với hệ số góc
$\sigma'$.

Lượt ngược đi qua đúng các ma trận đó (chuyển vị) và đúng các $\sigma'$ đó, nên gradient co cùng tỉ lệ:

In [ ]:
slope = float(np.mean([r[0] for r in rows]))
predict = slope * 0.05 * math.sqrt(128)
hidden = [g[k] / g[k + 1] for k in range(1, 6)]  # tầng 2–6, mỗi tầng so với tầng sau nó (cùng kích thước 128 × 128)
measured = float(np.exp(np.mean(np.log(hidden))))
print(f"dự đoán: σ′ · std · √128 = {vn(slope, 3)} · 0,05 · {vn(math.sqrt(128), 2)} = {vn(predict, 3)}")
print(f"đo được: mỗi tầng nhân gradient với {vn(measured, 3)} (trung bình nhân của tầng 2 → 6)")
assert all(r[1] == 0 for r in rows) and min(r[0] for r in rows) > 0.2, "khác với reel"
assert abs(math.log(measured / predict)) < math.log(1.5)

Hai con số khớp nhau. Vậy gradient biến mất ở đây là do **trọng số khởi tạo nhỏ** làm mỗi tầng nhân cả tín hiệu lẫn
gradient với khoảng 0,14, chứ không phải do sigmoid bão hòa. Đạo hàm sigmoid ≤ 0,25 là một phần của thừa số đó,
nhưng một mình nó không quyết định: với trọng số lớn hơn, $\text{std} \cdot \sqrt{128}$ có thể bù lại (Bài 09).

## 5 · Khởi tạo: nhỏ quá, rồi lớn quá

Tài liệu thử khởi tạo với std = 1 và std = 10. Ta quét rộng hơn, ngay lúc khởi tạo (chưa huấn luyện): mỗi giá trị
std, một lô 512 ảnh, một lượt xuôi và một lượt ngược.

### 🤔 Dự đoán trước
Std càng lớn thì gradient ở tầng 1 càng lớn. Vậy std 10 có học tốt hơn std 1 không?

In [ ]:
def at_init(std):
    set_seed(0)
    m = MLP(784, 128, 10, std=std)
    X, y = X_all[:512], y_all[:512]
    nn.CrossEntropyLoss()(m(X), y).backward()
    gr = [layer.weight.grad.abs().mean().item() for layer in linears(m)]
    return gr[0] / gr[-1], float(np.mean([r[0] for r in probe(m, X)]))


stds = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 3.0, 10.0]
scan = [at_init(s) for s in stds]
print("std     tầng 1 / tầng ra   σ′ trung bình")
for s, (r, d) in zip(stds, scan):
    print(f"{vn(s, 2):>5}   {num(r):>16}   {vn(d, 3):>12}")

fig, ax1 = plt.subplots(figsize=(7, 3.4))
ax1.plot(stds, [r for r, _ in scan], "o-", color=BLUE, label="gradient tầng 1 / tầng ra")
ax1.axhline(1, color=GREY, lw=1, ls=":")
ax1.set(xscale="log", yscale="log", xlabel="std khởi tạo", ylabel="tầng 1 / tầng ra")
ax2 = ax1.twinx()
ax2.plot(stds, [d for _, d in scan], "s--", color=ORANGE, label="σ′ trung bình")
ax2.set_ylabel("σ′ trung bình")
ax2.set_ylim(0, 0.27)
ax2.spines["right"].set_visible(True)
fig.legend(loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.02))
vn_axes(fig)
plt.show()

In [ ]:
for k in ("baseline", "std1", "std10"):
    r = RUNS["runs"][k]
    print(f"{k:9} test sau 100 epoch, 3 seed: {', '.join(pct(v) for v in r['test'])}")

Đường xanh (chấm tròn, trục trái) là gradient tầng 1 so với tầng ra; nó tăng theo std và vượt 1 (vạch chấm) ở
giữa std 1 và std 3. Đường cam (ô vuông, nét đứt, trục phải) là $\sigma'$ trung bình; nó giảm khi std tăng, vì $z$ lớn đẩy
sigmoid ra hai đầu phẳng. Hai đường đi ngược nhau, và không std nào giữ được cả hai.

Std 1 học được (khoảng 73–74%). Std 10 làm gradient tầng 1 *lớn hơn* tầng ra, nhưng giờ sigmoid thực sự bão hòa:
$\sigma'$ chỉ còn khoảng 0,005, và mạng chỉ hơn đoán mò một chút (10–20% tùy seed). Đó mới đúng là "bão hòa đạo hàm" mà tài liệu tả, và nó do
trọng số lớn gây ra, không phải trọng số nhỏ. Lối ra thật là chọn std theo số đầu vào (Xavier, He: Bài 09) hoặc
giữ tín hiệu ở đúng thang đo trong suốt quá trình huấn luyện (BatchNorm: mục sau).

## 6 · BatchNorm, và "Shortcut Back" có và không có hook

BatchNorm (Bài 10) chuẩn hóa $z$ của mỗi tầng về trung bình 0, độ lệch 1 trên từng lô, trước sigmoid. Mã của tài liệu:

In [ ]:
class MLP_BN(nn.Module):
    def __init__(self, input_dims, hidden_dims, output_dims):
        super().__init__()
        dims = [input_dims] + [hidden_dims] * 7
        self.layers = nn.ModuleList([nn.Linear(a, hidden_dims) for a in dims[:-1]])
        self.bns = nn.ModuleList([nn.BatchNorm1d(hidden_dims) for _ in range(7)])
        self.output = nn.Linear(hidden_dims, output_dims)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.05)
                nn.init.constant_(module.bias, 0.0)

    def forward(self, x):
        x = nn.Flatten()(x)
        for layer, bn in zip(self.layers, self.bns):
            x = nn.Sigmoid()(bn(layer(x)))
        return self.output(x)


loaders = make_loaders()
model = MLP_BN(784, 128, 10)
bn = train(model, optim.SGD(model.parameters(), lr=1e-2), loaders, epochs=1)
assert abs(bn[0]["train_loss"] - 1.9205440924) < 1e-3, "khác với reel"

Tài liệu viết BatchNorm thành bảy dòng `self.bn1 … self.bn7`; ở đây gom vào `ModuleList` cho gọn. Thứ tự tạo các
tầng giống hệt, nên trọng số khởi tạo cũng giống hệt, và epoch đầu ra đúng như lần chạy dài: ngay sau một epoch,
mạng đã đoán đúng khoảng 68% ảnh kiểm định.

**"Shortcut Back"** là cách cuối cùng của tài liệu. Nó gắn vào mỗi tầng ẩn một `ShortcutBackLayer`: một *hook* nhân
gradient đi ngược qua tầng đó với $1 + \alpha/2$, với $\alpha$ tăng theo độ sâu và giảm dần qua các epoch. Nhưng
chính docstring trong mã của tài liệu ghi rằng mô hình này còn đổi ba thứ khác: ReLU thay sigmoid, thêm BatchNorm, và
khởi tạo Kaiming. Muốn biết hook đóng góp bao nhiêu, ta chạy cùng mô hình với $\alpha = 0$: hook nhân với đúng 1.

In [ ]:
class ShortcutBackLayer(nn.Module):
    """Hook nhân gradient đi ngược qua tầng với 1 + 0,5·α (mã của tài liệu, rút gọn chú thích)."""

    def __init__(self, layer_idx, num_layers, alpha_init=1.0):
        super().__init__()
        self.layer_idx, self.num_layers = layer_idx, num_layers
        self.alpha = alpha_init * (layer_idx + 1) / num_layers
        self.register_buffer("grad_scale", torch.tensor(1.0))

    def forward(self, x):
        if self.training:
            x.register_hook(lambda grad: grad * (1.0 + self.alpha * 0.5))
        return x

    def update_alpha(self, epoch, max_epochs):
        a0 = 1.0 * (self.layer_idx + 1) / self.num_layers
        a1 = 0.1 * (self.layer_idx + 1) / self.num_layers
        self.alpha = a0 * (a1 / a0) ** (epoch / max(max_epochs, 1))


class ShortcutBackMLP(nn.Module):
    def __init__(self, input_dims, hidden_dims, output_dims, alpha_init=1.0):
        super().__init__()
        self.num_hidden_layers = 7
        self.input_layer = nn.Linear(input_dims, hidden_dims)
        self.hidden_layers = nn.ModuleList([nn.Linear(hidden_dims, hidden_dims) for _ in range(6)])
        self.output_layer = nn.Linear(hidden_dims, output_dims)
        self.shortcut_back_layers = nn.ModuleList([ShortcutBackLayer(i, 7, alpha_init) for i in range(7)])
        self.batch_norm_layers = nn.ModuleList([nn.BatchNorm1d(hidden_dims) for _ in range(7)])
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = nn.Flatten()(x)
        layers = [self.input_layer, *self.hidden_layers]
        for i, layer in enumerate(layers):
            x = self.shortcut_back_layers[i](nn.ReLU()(self.batch_norm_layers[i](layer(x))))
        return self.output_layer(x)


def shortcut_run(hook, epochs=1):
    loaders = make_loaders()
    model = ShortcutBackMLP(784, 128, 10)
    if not hook:
        for layer in model.shortcut_back_layers:
            layer.alpha = 0.0
    before = (lambda e: [layer.update_alpha(e, 100) for layer in model.shortcut_back_layers]) if hook else None
    return train(model, optim.SGD(model.parameters(), lr=1e-2), loaders, epochs, before_epoch=before)


print("có hook:")
with_hook = shortcut_run(True)
print("không hook (α = 0):")
no_hook = shortcut_run(False)
assert abs(with_hook[0]["train_loss"] - 0.9759439719) < 1e-3 and abs(no_hook[0]["train_loss"] - 1.4839382683) < 1e-3
gain = with_hook[0]["grad"][0] / no_hook[0]["grad"][0]
print(
    f"\nepoch 1, gradient tầng 1: có hook {num(with_hook[0]['grad'][0])} · không hook "
    f"{num(no_hook[0]['grad'][0])} (gấp {vn(gain, 1)} lần)"
)

Lúc đầu, hook có tác dụng rõ: gradient tầng 1 lớn gấp hơn ba lần, và loss của epoch đầu thấp hơn hẳn. Tài liệu bảo
gọi `update_alphas(epoch, 100)` nhưng không nói gọi lúc nào; ta gọi nó ở đầu mỗi epoch, nên epoch đầu chạy với
$\alpha$ lớn nhất.

Lớp tên `ShortcutBackLayer` của tài liệu cũng tự ghi rằng nó không cài phương pháp của bài báo *Take A Shortcut Back*
(Guo và cộng sự, NeurIPS 2024). Bài báo đó thêm các nhánh nối tầng giữa với đầu ra lúc huấn luyện, cho mạng nơ-ron
xung (spiking); ở đây chỉ có phép nhân gradient.

### 🤔 Dự đoán trước
Sau 100 epoch, mô hình có hook hơn mô hình không hook bao nhiêu điểm phần trăm: khoảng 0, khoảng 1, hay khoảng 10?

## 7 · Cả 15 thí nghiệm: 100 epoch × 3 seed

Mỗi thí nghiệm chạy lại đúng mã của tài liệu, với seed 42 (của tài liệu), 43 và 44. Seed quyết định cách chia
90/10, trọng số khởi tạo và thứ tự các lô; độ chênh giữa ba seed cho biết khác biệt nào là thật. Những chỗ mã in ra
không chạy được nguyên văn, hoặc tài liệu không ghi tham số, được sửa ít nhất có thể:

- **Mọi thí nghiệm:** `batch_size = 512`, giá trị trong notebook lời giải.
- **Chuẩn hóa gradient:** lớp này đặt sau mỗi tầng `Linear` ẩn, trước sigmoid (tài liệu chỉ nói "sau các tầng Linear").
- **Huấn luyện từng tầng:** 15 epoch cho mỗi giai đoạn trong 7 giai đoạn (tài liệu không ghi), và thêm một lần chạy
  50 epoch mỗi giai đoạn.
- **Shortcut Back:** SGD với `lr = 1e-2` (tài liệu không ghi bộ tối ưu), gọi `update_alphas` ở đầu mỗi epoch; thêm
  lần chạy $\alpha = 0$ ở trên.
- **Huấn luyện tăng dần:** tài liệu gọi `get_newly_unfrozen_params` mà không định nghĩa nó; ta định nghĩa là các tham
  số của tầng vừa mở. Bộ tối ưu chỉ nhận các tham số đang huấn luyện, vì `add_param_group` báo lỗi khi nhận lại một
  tham số nó đã có. Tài liệu không ghi bộ tối ưu: ta dùng SGD `lr = 1e-2` như mọi thí nghiệm khác, và thêm một lần
  chạy với Adam (`lr = 1e-3`, tầng mới mở `2e-3`).

In [ ]:
LABELS = {
    "baseline": "gốc (sigmoid, std 0,05, SGD)",
    "std1": "khởi tạo std 1",
    "std10": "khởi tạo std 10",
    "relu": "ReLU",
    "adam": "Adam",
    "batchnorm": "BatchNorm",
    "mynorm": "chuẩn hóa tự viết",
    "skip": "kết nối tắt",
    "gradnorm": "chuẩn hóa gradient",
    "layerwise": "từng tầng · 15 epoch/giai đoạn",
    "layerwise_long": "từng tầng · 50 epoch/giai đoạn",
    "progressive": "tăng dần · SGD",
    "progressive_adam": "tăng dần · Adam",
    "shortcut": "Shortcut Back (có hook)",
    "shortcut_nohook": "… không hook",
}
order = sorted(LABELS, key=lambda k: np.mean(RUNS["runs"][k]["test"]))
for k in reversed(order):
    t = RUNS["runs"][k]["test"]
    paper = f"   (tài liệu in {pct(RUNS['paper'][k]['test'])})" if k in RUNS["paper"] else ""
    print(f"{LABELS[k]:31} {pct(min(t))} – {pct(max(t))}{paper}")
fig, ax = plt.subplots(figsize=(8, 5.4))
for i, k in enumerate(order):
    t = RUNS["runs"][k]["test"]
    ax.plot([min(t), max(t)], [i, i], color=GREY, lw=2)
    ax.scatter(
        t, [i] * 3, color=GREEN if np.mean(t) > 0.5 else ORANGE, marker="o" if np.mean(t) > 0.5 else "s", zorder=3
    )
ax.axvline(0.1, color=GREY, ls=":", lw=1)
ax.text(0.105, len(order) - 0.6, "đoán mò", color=GREY, fontsize=8)
ax.set_yticks(range(len(order)), [LABELS[k] for k in order])
ax.set(xlabel="độ chính xác trên tập kiểm tra sau 100 epoch (3 seed)", xlim=(0, 1))
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
plt.show()

Mỗi hàng là một thí nghiệm, xếp từ thấp lên cao; ba điểm là ba seed, nối bằng một vạch xám. Điểm xanh tròn: thoát
hẳn mức đoán mò; ô vuông cam: dưới 50%. Vạch chấm là đoán mò (10%).

- **Hiệu quả:** BatchNorm, tăng dần với Adam và Shortcut Back đứng đầu (khoảng 86–88%); Adam, chuẩn hóa tự viết, ReLU
  và khởi tạo std 1 theo sau. Kết nối tắt của tài liệu chỉ nối qua hai cụm tầng nên giúp ít hơn (khoảng 67%).
- **Kẹt ở 10%, chạy đúng mã in:** chuẩn hóa gradient (nó ép gradient ở mỗi tầng về độ lệch chuẩn 1 bất kể độ lớn
  thật; ô dưới cho thấy sau epoch đầu, $\sigma'$ ở mọi tầng đã gần 0), huấn luyện từng tầng với 15 epoch mỗi giai
  đoạn, và huấn luyện tăng dần với SGD.
  Mỗi giai đoạn từng tầng tạo một tầng ra mới từ đầu; cho 50 epoch mỗi giai đoạn, nó lên khoảng 43%.
- **Tăng dần:** tài liệu in 83,04% nhưng không ghi bộ tối ưu. Với SGD (như mọi thí nghiệm khác) nó kẹt; với Adam nó
  đạt khoảng 88%.

In [ ]:
gn = RUNS["runs"]["gradnorm"]["act1"]
print("chuẩn hóa gradient, sau epoch 1: σ′ trung bình mỗi tầng", ", ".join(num(v, 2) for v in gn["slope"]))
print("                                 tỉ lệ bão hòa", ", ".join(pct(v, 0) for v in gn["saturated"]))

In [ ]:
sb, nh = RUNS["runs"]["shortcut"]["test"], RUNS["runs"]["shortcut_nohook"]["test"]
print(
    f"Shortcut Back, có hook:  {', '.join(pct(v) for v in sb)}   (tài liệu in {pct(RUNS['paper']['shortcut']['test'])})"
)
print(f"Shortcut Back, không hook: {', '.join(pct(v) for v in nh)}")
print(
    f"chênh lệch trung bình: {vn(100 * (np.mean(sb) - np.mean(nh)), 2)} điểm · "
    f"có hook cao hơn ở cả 3 seed: {min(sb) > max(nh)}"
)
bn_test = RUNS["runs"]["batchnorm"]["test"]
print(f"sigmoid + BatchNorm (không ReLU, không Kaiming, không hook): {', '.join(pct(v) for v in bn_test)}")

Hook thêm khoảng 0,6 điểm, và lần này hơn ở cả ba seed. Nhưng bước nhảy từ 10% lên 86% gần như toàn bộ đến từ ba thứ
đổi cùng lúc với nó: chỉ riêng sigmoid + BatchNorm đã đạt khoảng 87%. Bài học không nằm ở một kỹ thuật cụ thể, mà ở
cách làm thí nghiệm: **đổi một thứ mỗi lần**, và chạy nhiều seed trước khi tin một chênh lệch nhỏ.

## 8 · Thử thay đổi

Ô dưới huấn luyện mạng gốc với std khác, 2 epoch. Thử 0,05, 0,2, 0,5, 1: std nào cho độ chính xác kiểm định cao
nhất? So với kết quả quét ở mục 5, điều đó có khớp với "gradient tầng 1 đủ lớn mà sigmoid chưa bão hòa"?

In [ ]:
STD = 0.5  # sửa ở đây
loaders = make_loaders()
model = MLP(784, 128, 10, std=STD)
trial = train(model, optim.SGD(model.parameters(), lr=1e-2), loaders, epochs=2)
print(f"std {vn(STD, 2)}: tầng 1 / tầng ra ở epoch cuối = {num(trial[-1]['grad'][0] / trial[-1]['grad'][-1])}")

## 9 · Giới hạn

- Một bộ dữ liệu, một kiến trúc, 100 epoch trên CPU. Trên GPU, các chữ số cuối sẽ khác (cộng theo thứ tự khác), nhưng
  kết luận không đổi.
- Tài liệu không ghi vài tham số (số epoch mỗi giai đoạn khi huấn luyện từng tầng, bộ tối ưu của Shortcut Back và của
  huấn luyện tăng dần); những gì ta chọn được in ở mục 7, và một lựa chọn khác có thể cho số khác.
- 3 seed đủ để thấy chênh lệch lớn; với chênh lệch dưới 1 điểm, nên chạy thêm.

## Tóm tắt
- Mạng gốc (sigmoid, std 0,05, SGD) đoán đúng 10% sau 100 epoch ở cả 3 seed, như tài liệu in; gradient tầng 1 nhỏ hơn
  tầng ra khoảng hai triệu lần.
- Thủ phạm là trọng số nhỏ, không phải bão hòa: $\sigma' \approx 0{,}24$ ở mọi tầng, và mỗi tầng nhân gradient với
  khoảng $\sigma' \cdot 0{,}05 \cdot \sqrt{128} \approx 0{,}14$.
- Std lớn chữa gradient nhưng làm sigmoid bão hòa: std 10 gần như đoán mò.
- BatchNorm, ReLU, Adam chữa được; chuẩn hóa gradient, huấn luyện từng tầng và tăng dần với SGD, chạy đúng mã in, vẫn
  kẹt ở 10%.
- "Shortcut Back" đổi bốn thứ cùng lúc; riêng hook chỉ thêm khoảng 0,6 điểm.

## Câu hỏi ôn tập
1. Vì sao $\sigma' \approx 0{,}24$ ở mọi tầng mà gradient vẫn biến mất?
2. Thừa số mỗi tầng $\sigma' \cdot \text{std} \cdot \sqrt{n}$ bằng bao nhiêu với std 1? Vì sao mạng std 1 vẫn học
   được dù thừa số đó khác 1?
3. Vì sao std 10 làm gradient tầng 1 lớn hơn tầng ra mà mạng vẫn không học?
4. Thí nghiệm "không hook" trả lời câu hỏi gì mà tài liệu không trả lời?
5. Huấn luyện từng tầng tạo một `nn.Linear(128, 10)` mới ở mỗi giai đoạn. Điều đó ảnh hưởng gì?

<details><summary>Gợi ý đáp án</summary>

1. Vì thừa số mỗi tầng còn nhân với độ lớn của trọng số: $0{,}24 \cdot 0{,}05 \cdot \sqrt{128} \approx 0{,}14 < 1$,
   nên gradient co theo cấp số nhân qua 7 tầng.
2. Khoảng $0{,}05 \cdot 1 \cdot 11{,}3 \approx 0{,}6$ ($\sigma'$ chỉ còn khoảng 0,05 vì nhiều nơ-ron bão hòa):
   gradient vẫn co, nhưng chậm hơn nhiều; tầng 1 nhận vài phần trăm gradient của tầng ra (mục 5) thay vì một phần
   triệu.
3. Với std 10, $|z|$ rất lớn nên $\sigma' \approx 0{,}005$: đầu ra của mỗi tầng gần như chỉ là 0 hoặc 1, và loss
   của epoch đầu rất lớn; gradient lớn nhưng không chỉ được hướng có ích.
4. Bao nhiêu phần kết quả đến từ hook, và bao nhiêu từ ReLU + BatchNorm + Kaiming đổi cùng lúc.
5. Mỗi giai đoạn bắt đầu với một tầng ra ngẫu nhiên, bỏ đi tầng ra đã học; các tầng mới thêm cũng khởi tạo std 0,05,
   nên mạng lại gặp đúng vấn đề ban đầu.
</details>

## Tìm hiểu thêm
- Nguyễn Thọ Anh Khoa, Nguyễn Phúc Thịnh, Dương Đình Thắng, Nguyễn Anh Khôi, *Tutorial: Vanishing Gradient*: tài liệu
  gốc của chuyên đề này.
- H. Xiao, K. Rasul & R. Vollgraf (2017), [Fashion-MNIST](https://arxiv.org/abs/1708.07747).
- X. Glorot & Y. Bengio (2010), [Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a.html).
- K. He và cộng sự (2015), [Delving Deep into Rectifiers](https://arxiv.org/abs/1502.01852) (khởi tạo Kaiming).
- S. Ioffe & C. Szegedy (2015), [Batch Normalization](https://arxiv.org/abs/1502.03167).
- Y. Guo và cộng sự (2024), [Take A Shortcut Back](https://arxiv.org/abs/2401.04486) (NeurIPS 2024).

**Quay lại:** [Bài 09 · Gradient biến mất & khởi tạo](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vanishing-gradient/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · torch {torch.__version__} · numpy {np.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)